# 02 · Run the experiment (Lab 7)

Runs the repository's own pipeline on Kaggle (GPU T4 x2, Internet on): install the pinned engine, clone the
repository, fetch and checksum MedQA, run all CPU tests, generate (greedy, 10 sampled, greedy rerun), evaluate.

**`SPLIT = "debug"`** checks the real vLLM code path on the 25 debug items. **`SPLIT = "test"`** runs the
633 test items and is refused by `run_generation.py` unless `configs/protocol_v1.yaml` says `status: FROZEN`.
Outputs in `/kaggle/working/`: `raw/<split>/*.jsonl` (quote item text; keep private), `eval/<split>/results.csv`,
`item_level.csv`, `figures/`.

In [ ]:
SPLIT = "debug"          # "debug" or "test"
COMMIT = "main"          # branch or exact commit hash to run
import subprocess, sys
sh = lambda c: print(subprocess.run(c, shell=True, capture_output=True, text=True, executable="/bin/bash").stdout[-6000:])

In [ ]:
# Install the pinned engine (requirements-gpu.txt) and fail fast if it cannot be imported
sh("pip install -q vllm==0.32.0 2>&1 | tail -3; pip uninstall -y -q torchaudio torchao 2>&1 | tail -2")
sh("python -c \"import vllm, torch; from vllm import LLM, SamplingParams; print('vllm', vllm.__version__, '| torch', torch.__version__, '| LLM import OK')\" 2>&1")

In [ ]:
# Code and data
sh(f"rm -rf /kaggle/working/repo && git clone -q https://github.com/vibhum-phd/lab7-oncology-llm-voting.git /kaggle/working/repo "
   f"&& cd /kaggle/working/repo && git checkout -q {COMMIT} && git log -1 --format='commit %H %s' && grep '^status' configs/protocol_v1.yaml")
sh("cd /kaggle/working/repo && python scripts/fetch_medqa.py data/raw/medqa_4opt_hf 2>&1 | grep -v Warning")

In [ ]:
# All CPU tests must pass before any generation
sh("cd /kaggle/working/repo && for t in tests/test_*.py; do echo -n \"$t: \"; python -X utf8 $t 2>&1 | tail -1; done")

In [ ]:
# Generate (resumable) and evaluate
sh(f"cd /kaggle/working/repo && python -X utf8 scripts/run_generation.py --split {SPLIT} --out /kaggle/working/raw/{SPLIT} 2>&1 | grep -E '^\\{{|refusing|Error|error' | tail -20")
sh(f"cat /kaggle/working/raw/{SPLIT}/run_summary.json")
sh(f"cd /kaggle/working/repo && python -X utf8 scripts/run_evaluation.py --split {SPLIT} --raw /kaggle/working/raw/{SPLIT} --out /kaggle/working/eval/{SPLIT} 2>&1 | tail -5")
sh("pip freeze > /kaggle/working/pip_freeze.txt; nvidia-smi --query-gpu=name,memory.used --format=csv")